# EkaQuant Evaluation Notebook
This notebook evaluates **Llama-3.2-3B-Instruct** using the **eka-eval** framework and prepares the environment for **EkaQuant** task-aware selective quantization experiments.

In [ ]:
import os
from datetime import datetime
from kaggle_secrets import UserSecretsClient

# 1. Hugging Face Authentication
# Ensure you have a Kaggle Secret named 'HF_TOKEN'
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
    else:
        print("Warning: HF_TOKEN not found in Kaggle Secrets.")
except Exception as e:
    print(f"Could not authenticate with HF_TOKEN: {e}")

In [ ]:
# 2. Install Dependencies
!pip install -q torch==2.10.0 transformers bitsandbytes accelerate peft 
!pip install -q numpy scipy kneed scikit-image tqdm

# 3. Install eka-eval from IIT Gandhinagar
!git clone https://github.com/lingo-iitgn/eka-eval.git
%cd eka-eval
!pip install -e .
%cd ..

# 4. Install EkaQuant
!git clone https://github.com/TarunNagarajan/EkaQuant.git
%cd EkaQuant
!pip install -e .
%cd ..

In [ ]:
# 5. Run Evaluation Baselines
model_id = "meta-llama/Llama-3.2-3B-Instruct"
model_name = model_id.split("/")[-1]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 8-bit Evaluation
out_8bit = f"./results/{model_name}_8bit_{timestamp}"
print(f"Starting 8-bit evaluation -> {out_8bit}")
!eka-eval \
    --model {model_id} \
    --tasks "mmlu_in_hi" \
    --load_in_8bit True \
    --output_path {out_8bit}

# 4-bit Evaluation
out_4bit = f"./results/{model_name}_4bit_{timestamp}"
print(f"Starting 4-bit evaluation -> {out_4bit}")
!eka-eval \
    --model {model_id} \
    --tasks "mmlu_in_hi" \
    --load_in_4bit True \
    --output_path {out_4bit}

In [ ]:
# 6. Summary of Results
import glob
import pandas as pd

csv_files = glob.glob("./results/**/summary.csv", recursive=True)
if not csv_files:
    print("No results found yet.")
for f in csv_files:
    print(f"\n--- {f} ---")
    df = pd.read_csv(f)
    display(df)